# RAG 과제

## 0. 환경 세팅

In [ ]:
import os
os.environ["RAG_DATA_DIR"] = "/content/data"

from google.colab import drive; drive.mount("/content/drive")

!git clone -q <repo-url>
!pip install -e .
%load_ext autoreload
%autoreload 2

## 1. 데이터 다운로드

In [ ]:
from huggingface_hub import snapshot_download

REPO_ID = "NLPlab-skku/RAG-assignment"
snapshot_download("NLPlab-skku/RAG-assignment", repo_type="dataset",
                  revision="v1", local_dir="data")

## 2. Config 설정

In [18]:
from rag_assignment.paths import OUTPUTS_DIR


### Retriever
RETRIEVER_MODE = "lexical"            # "lexical" or "dense"
RETRIEVAL_K = 100

### Generator
GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"
CONTEXT_K = 5
USE_CONTEXT = True
MAX_SEQ_LEN = 2048
MAX_ANS_LEN = 32
BATCH_SIZE = 8
USE_FP16 = True

### Result
OUTPUT_DIR = OUTPUTS_DIR / f"{RETRIEVER_MODE}_Rk={RETRIEVAL_K}_Ck={CONTEXT_K}_{USE_CONTEXT}"

## 3. Module 준비
- Retriever와 Generator 준비

### Load retriever

In [19]:
from rag_assignment.utils import load_corpus
from rag_assignment.paths import DATA_DIR, INDEX_DIR

corpus = load_corpus(DATA_DIR / "corpus.jsonl")
index_dir = INDEX_DIR / RETRIEVER_MODE

Load /workspaces/assignment/student/data/corpus.jsonl...: 1087303it [00:07, 137013.16it/s]


In [20]:
from rag_assignment.retriever import get_retriever

retriever = get_retriever(RETRIEVER_MODE, corpus, index_dir)

Load lexical index from /workspaces/assignment/student/data/index/lexical


### Load generator

In [12]:
from rag_assignment.generator import HFGenerator

generator = HFGenerator(
    model_name_or_path=GENERATOR_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    max_ans_length=MAX_ANS_LEN,
    batch_size=BATCH_SIZE,
    fp16=USE_FP16
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## 4. Warmup
* 초반 시간 측정의 부정확함 방지

### Load Dev QA

In [21]:
from rag_assignment.utils import load_qa_dataset

dev_qa = load_qa_dataset(DATA_DIR / "dev_qa.jsonl")
qa = dev_qa[0]
docs = [r.doc for r in retriever.retrieve(qa.question, CONTEXT_K)]
for d in docs:
    print(d.pid, d.title, "|", d.text[:80])
out = generator.generate(qa.question, docs)

print("Q:", qa.question, "| gold:", qa.answers)
print("raw:", repr(out.raw_text), "-> pred:", out.text)

Load /workspaces/assignment/student/data/dev_qa.jsonl...: 1000it [00:00, 142770.24it/s]

20926585 Idea Popular de Guinea Ecuatorial | Idea Popular de Guinea Ecuatorial The Idea Popular de Guinea Ecuatorial (IPGE) w
20989835 TVGE | TVGE Televisión de Guinea Ecuatorial (TVGE, translated into english as "Televisi
20926595 Idea Popular de Guinea Ecuatorial | 1979, and began interviewing witnesses and collecting evidence against the Macía
120670 History of Equatorial Guinea | groups who had taken refuge from General Franco's paternal hand in Cameroun and 
20926592 Idea Popular de Guinea Ecuatorial | announced from Acra that they had joined forces in a new Frente Nacional y Popul
Q: where is guinea ecuatorial located on the map | gold: ['in Central Africa']
raw: 'Africa' -> pred: Africa


## 5. 실행

In [22]:
from rag_assignment.pipeline import RAGPipeline

pipeline = RAGPipeline(retriever, generator, RETRIEVAL_K, CONTEXT_K)
pipeline.run(dev_qa, use_context=USE_CONTEXT, output_dir=OUTPUT_DIR)

Split strings:   0%|          | 0/1000 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1000 [00:00<?, ?it/s]

Generate: 100%|██████████| 125/125 [02:07<00:00,  1.02s/it]


[GenerationResult(text='Africa', raw_text='Africa', n_input_tokens=859, n_output_tokens=2),
 GenerationResult(text='1587', raw_text='1587', n_input_tokens=875, n_output_tokens=5),
 GenerationResult(text='13', raw_text='13', n_input_tokens=835, n_output_tokens=3),
 GenerationResult(text='Eddie Rabbitt', raw_text='Eddie Rabbitt', n_input_tokens=846, n_output_tokens=6),
 GenerationResult(text='No', raw_text='No', n_input_tokens=867, n_output_tokens=2),
 GenerationResult(text='Hazard, Kentucky', raw_text='Hazard, Kentucky', n_input_tokens=791, n_output_tokens=5),
 GenerationResult(text='1979', raw_text='1979', n_input_tokens=833, n_output_tokens=5),
 GenerationResult(text='Salem', raw_text='Salem', n_input_tokens=781, n_output_tokens=3),
 GenerationResult(text='Alvin, Simon, Theodore', raw_text='Alvin, Simon, Theodore', n_input_tokens=910, n_output_tokens=7),
 GenerationResult(text='2017', raw_text='2017', n_input_tokens=823, n_output_tokens=5),
 GenerationResult(text='soul music, jazz, an

## 6. 평가

In [23]:
from rag_assignment.grading.evaluate import evaluate, print_summary

summary, results = evaluate(OUTPUT_DIR, dev_qa, corpus)
print_summary(summary)

Evaluate: 100%|██████████| 1000/1000 [00:04<00:00, 233.72it/s]

#questions : 1000
span-EM    : 0.3740
F1         : 0.3948
recall@k   : @1=0.2680  @5=0.4910  @10=0.6150  @20=0.7320  @100=0.8920
time       : retrieval=10.7s  generation=127.4s  (total 138.0s on NVIDIA RTX A6000)


## 7. 오답 분석
- 확인용 오답 분석

In [17]:
hit_but_wrong = [r for r in results if r.metrics.recall_at_k[CONTEXT_K] and not r.metrics.span_em]
miss = [r for r in results if not r.metrics.recall_at_k[CONTEXT_K]]
print(f"검색 성공·생성 실패 {len(hit_but_wrong)} / 검색 실패 {len(miss)}")
for r in hit_but_wrong[:5]:
    print(r.question, "| pred:", r.pred_answer, "| gold:", r.answers)

검색 성공·생성 실패 244 / 검색 실패 246
when did the three point shot start in basketball | pred: October 12, 1979 | gold: ['1961']
when was the theory of seafloor spreading proposed | pred: 1960 | gold: ['1960s']
when did take your daughter to work start | pred: 1993 | gold: ['April 22 , 1993']
how many episodes does riverdale season one have | pred: 22 | gold: ['13']
the age in which early humans made tools is called the | pred: Paleolithic | gold: ['The Stone Age']
